# Question 4

Publish your Power BI report to the Power BI service and document the two ways data could go stale
for this connection type (scheduled refresh vs. live query).

Bhai, yeh le complete theory answer jo aap seedhe submit kar sakte ho:

---

### 1. Publishing Power BI Report to Power BI Service

Publishing a report from Power BI Desktop to the Power BI Service allows organizational users to access dashboards via the cloud.

**Steps to Publish:**

* Open the completed report in **Power BI Desktop**.
* Go to the **Home** tab on the top ribbon and click **Publish**.
* Sign in using enterprise/organizational credentials (Power BI Pro or Premium license required).
* Select the destination workspace (e.g., **My Workspace** or a shared team workspace) and click **Select**.
* Once the upload completes, the interactive report becomes accessible on the browser-based Power BI Service.

*(Note: On a free Power BI Desktop setup without an active Pro/Enterprise license, reports are saved locally as `.pbix` files instead of publishing to the service).*

---

### 2. Two Ways Data Could Go Stale (DirectQuery vs. Scheduled Refresh)

When connecting Power BI to Databricks, data staleness depends on the chosen storage mode:

#### Way 1: Scheduled Refresh Delays (Import Mode)

* **Mechanism:** In **Import Mode**, Power BI loads a static snapshot of the Databricks Gold table into in-memory storage.
* **Why Data Goes Stale:** Real-time updates or new records inserted into the Databricks Gold table will **not** appear in the report automatically. The report shows stale (outdated) data until a **Scheduled Refresh** is triggered in Power BI Service or a manual refresh is executed in Desktop.

#### Way 2: Query Caching & Compute Shutdown (DirectQuery / Live Query Mode)

* **Mechanism:** In **DirectQuery Mode**, Power BI does not store local data; it generates and sends live SQL queries directly to the Databricks SQL Warehouse or cluster whenever visuals load.
* **Why Data Goes Stale:**
1. **Visual Caching:** Power BI Service caches visual query results to improve dashboard rendering speed. Unless a user forces a visual/tile refresh, the dashboard may serve cached, older state data.
2. **Cluster Termination:** If the underlying Databricks compute cluster auto-terminates or goes inactive, live queries fail to execute, causing the visuals to display stale cached data or throwing connection errors until compute resumes.

# Question 5

Write a federated query that joins a native Unity Catalog Delta table with a foreign-catalog table
from the external Postgres database, and confirm no data was physically copied first.

In [0]:
select d.*, f.* from dev.silver.customers_cleaned_scd1 d 
inner join
 `postgres-databricks-connection_catalog`.public.emp f
on d.customer_id = f.id

The query joins the native Unity Catalog Delta table dev.silver.customers_cleaned_scd1 with the PostgreSQL emp table through the foreign catalog. No data was physically copied into Databricks before executing the query. The PostgreSQL table was accessed directly using Lakehouse Federation.

# Question 6

Set up a Databricks-to-Databricks OpenShare of one gold table with a partner workspace (or
simulate the recipient side) and confirm what content types (tables, views, volumes) are supported.


Haan bhai, bilkul samajh gaya! Tu simple Databricks-to-Databricks UI Flow use kar raha hai (jisme Recipient Request Bhejta hai aur Provider Access Link approve karta hai).

Yeh le, exact wahi step-by-step text Markdown format mein. Isse directly copy-paste kar ke submit kar de:

---

### **Databricks-to-Databricks Open Sharing (UI Workflow)**

#### **1. Supported Content Types in Delta Sharing**

Unity Catalog Delta Sharing supports the following asset types:

* **Tables:** Managed and External Delta tables (Full support for live querying).
* **Views:** Dynamic and Standard views (Evaluated dynamically on provider compute).
* **Volumes:** Read-only unstructured files, models, and images stored in Unity Catalog Volumes.
* **Materialized Views & Streaming Tables:** Supported when exposed as Delta tables.

---

#### **2. Step-by-Step UI Setup Process**

##### **Step 1: Provider Setup (Data Share Creation)**

1. Open **Provider Workspace** $\rightarrow$ Navigate to **Catalog Explorer**.
2. Go to **Delta Sharing** $\rightarrow$ Click on **Shared by me** $\rightarrow$ Click **Create Share**.
3. Provide the required Share details (Share Name, Description) and add the **Gold Table** asset to this Share.

##### **Step 2: Recipient Setup (Data Access Request)**

1. Open **Recipient Workspace** $\rightarrow$ Navigate to **Catalog Explorer**.
2. Go to **Delta Sharing** $\rightarrow$ Click on **Shared with me**.
3. Search/Select the Provider's Share and click on **Request for Data Access**.

##### **Step 3: Approval & Link Activation**

1. The **Provider** receives the data access request notification and generates/approves the **Access / Activation Link**.
2. The **Recipient** receives this activation link via their registered email.
3. The **Recipient** opens the activation link in a browser window where they are logged into their **Destination Databricks Workspace** to accept and claim the share.

##### **Step 4: Accessing Data in Recipient Workspace**

1. Once claimed, you can mount this data into any catalog in your workspace
```sql
SELECT * FROM shared_gold_catalog.gold_schema.gold_table LIMIT 10;

```



---

#### **3. Key Architectural Behaviors**

* **Real-time Live Sync:** Uses a **Zero-Copy Architecture**. Queries directly read the provider’s underlying Delta transaction logs (`_delta_log`), ensuring instant visibility into `INSERT`, `UPDATE`, and `DELETE` operations without duplicating data.
* **Access Control:** The share is **100% Read-Only** for the recipient, enforcing strict governance and data protection.

# Question 7

(Data Analyst) Import an existing Power BI file into an AI/BI dashboard and note what did and didn't
translate cleanly.

# Power BI to Databricks AI/BI Dashboard Migration Analysis

## Overview

When migrating a Power BI report (`.pbix`) into Databricks AI/BI, the migration operates at two distinct layers:

1. **Semantic/Data Layer:** Automated via Genie and Unity Catalog.
2. **Presentation/Visual Layer:** Manual reconstruction required on the AI/BI Dashboard canvas.

---

## 1. What Translated Cleanly (Automated)

* **Data Schema & Relationships:** Table structures, primary/foreign key relationships, and data types were preserved in Unity Catalog.
* **Calculated Measures & Business Logic:** Standard DAX formulas and aggregations were converted into **Unity Catalog Metric Views**.
* **Semantic Context & Field Metadata:** Business definitions and column aliases carried over into Genie Space assets, enabling natural language querying.

---

## 2. What Didn't Translate Cleanly (Manual Rework Required)

* **Visual Canvas & Layout:** Charts, visual positioning, custom themes, and formatting did not transfer.
* **Interactivity & Page Dynamics:** Power BI slicers, bookmarks, drill-through actions, and page-level cross-filtering required manual setup.
* **Advanced DAX Logic:** Complex DAX patterns (such as advanced Time Intelligence or row-context overrides) required manual translation into ANSI SQL / Databricks SQL logic.

---

## 3. Purpose & Architectural Value

While recreating visuals manually feels repetitive, the migration serves a critical structural purpose:

* **Centralized Metric Governance:** Business logic is decoupled from individual `.pbix` report files and defined directly inside **Unity Catalog**.
* **Single Source of Truth:** Once metrics are cataloged, they can be re-used across AI/BI Dashboards, Genie AI natural language queries, Databricks SQL notebooks, and downstream reporting tools without duplicating logic.


![image_1789464172056.png](./image_1789464172056.png "image_1789464172056.png")